In [1]:
import numpy as np
import galsim
from lsst.daf.butler import Butler
from lsst.summit.utils import ConsDbClient
from lsst.ts.ofc import OFCData, SensitivityMatrix
from astropy.table import Table, QTable, vstack
from pathlib import Path

ModuleNotFoundError: No module named 'lsst.ts.ofc'

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
output_dir = Path('dk_to_dof_results')

# General obs and  info

In [ ]:
n_focal, n_pupil = 6, 21

In [ ]:
pupil_indices = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,22,23,24,25,26]
focal_indices = np.arange(1,7)

In [ ]:
day_obs_vals = [20260315, 20260316, 20260317]

In [ ]:
dof_labels = [ "M2_hex_z", "M2_hex_x", "M2_hex_y", "M2_hex_rx", "M2_hex_ry", "Cam_hex_z", "Cam_hex_x", "Cam_hex_y", "Cam_hex_rx", "Cam_hex_ry", ] 
dof_labels += [f"M1M3_B{i}" for i in range(1, 21)] 
dof_labels += [f"M2_B{i}" for i in range(1, 21)]

### Load and trim sensitivity matrix

In [ ]:
ofc_data = OFCData('lsst')
ideal_sens_dz = galsim.zernike.DoubleZernike(
    ofc_data.sensitivity_matrix[..., :],
    # Rubin annuli
    uv_inner=ofc_data.config["field"]["radius_inner"],
    uv_outer=ofc_data.config["field"]["radius_outer"],
    xy_inner=ofc_data.config["pupil"]["radius_inner"],
    xy_outer=ofc_data.config["pupil"]["radius_outer"],
) # indexed [k, j, dof]

#### trying to understand....

In [ ]:
ideal_sens_dz.coef.shape

In [ ]:
with np.printoptions(linewidth=200):
    for i in range(10):
        print(dof_labels[i])
        print(ideal_sens_dz.coef[:7,:5,i])
        print()


In [ ]:
ideal_sens_dz.coef[1:,4:,:]

In [ ]:
ideal_sens_dz.coef[:,1,:]

In [ ]:
model_smatrix = ideal_sens_dz.coef[np.ix_(focal_indices, pupil_indices)][..., :]
A = model_smatrix.reshape(-1, 50)

In [ ]:
def test_reshape_indexing():
    """Test that reshaping preserves the correct indexing order."""
    
    # Create a test 3D array with known values
    n_focal = 3
    n_pupil = 4
    n_dof = 5
    
    # Fill with unique values so we can track them
    test_array = np.arange(n_focal * n_pupil * n_dof).reshape(n_focal, n_pupil, n_dof)
    
    print("Original 3D array shape:", test_array.shape)
    print("\nOriginal array:")
    print(test_array)
    
    # Reshape using your method
    A = test_array.reshape(-1, n_dof)
    print("\nReshaped to 2D, shape:", A.shape)
    print("\nReshaped array:")
    print(A)
    
    # Test 1: Check specific elements
    print("\n=== Test 1: Element tracking ===")
    print(f"Original [0, 0, 0] = {test_array[0, 0, 0]}")
    print(f"Reshaped [0, 0] = {A[0, 0]}")
    print(f"Match: {test_array[0, 0, 0] == A[0, 0]}")
    
    print(f"\nOriginal [0, 1, 0] = {test_array[0, 1, 0]}")
    print(f"Reshaped [1, 0] = {A[1, 0]}")
    print(f"Match: {test_array[0, 1, 0] == A[1, 0]}")
    
    print(f"\nOriginal [1, 0, 0] = {test_array[1, 0, 0]}")
    print(f"Reshaped [{n_pupil}, 0] = {A[n_pupil, 0]}")
    print(f"Match: {test_array[1, 0, 0] == A[n_pupil, 0]}")
    
    # Test 2: Verify the indexing formula
    print("\n=== Test 2: Indexing formula ===")
    print("Flattening order in reshape(-1, n_dof) is C-order (row-major)")
    print("Formula: A[k*n_pupil + j, i] = test_array[k, j, i]")
    
    all_match = True
    for k in range(n_focal):
        for j in range(n_pupil):
            for i in range(n_dof):
                flat_index = k * n_pupil + j
                original_val = test_array[k, j, i]
                reshaped_val = A[flat_index, i]
                if original_val != reshaped_val:
                    print(f"MISMATCH at k={k}, j={j}, i={i}: {original_val} != {reshaped_val}")
                    all_match = False
    
    print(f"All elements match: {all_match}")
    
    # Test 3: Reverse check - reconstruct original
    print("\n=== Test 3: Reverse reconstruction ===")
    reconstructed = A.reshape(n_focal, n_pupil, n_dof)
    print(f"Reconstructed shape: {reconstructed.shape}")
    print(f"Arrays are identical: {np.array_equal(test_array, reconstructed)}")
    
    # Test 4: With your actual dimensions
    print("\n=== Test 4: With actual dimensions (6, 21, 50) ===")
    n_focal_real = 6
    n_pupil_real = 21
    n_dof_real = 50
    
    real_test = np.random.randn(n_focal_real, n_pupil_real, n_dof_real)
    A_real = real_test.reshape(-1, n_dof_real)
    
    print(f"Original shape: {real_test.shape}")
    print(f"Reshaped shape: {A_real.shape}")
    print(f"Expected: ({n_focal_real * n_pupil_real}, {n_dof_real}) = (126, 50)")
    
    # Verify specific mappings
    k, j, i = 2, 10, 25  # arbitrary indices
    flat_idx = k * n_pupil_real + j
    print(f"\ntest_array[k={k}, j={j}, i={i}] = {real_test[k, j, i]:.6f}")
    print(f"A[{flat_idx}, {i}] = {A_real[flat_idx, i]:.6f}")
    print(f"Match: {np.isclose(real_test[k, j, i], A_real[flat_idx, i])}")
    
    return test_array, A


def test_with_labeled_sensitivity():
    """Test with a sensitivity matrix that has interpretable structure."""
    
    n_focal = 6
    n_pupil = 21
    n_dof = 50
    
    # Create sensitivity matrix where each layer has a unique constant
    sensitivity = np.zeros((n_focal, n_pupil, n_dof))
    for k in range(n_focal):
        sensitivity[k, :, :] = k * 100  # Each focal layer has value k*100
    
    # Add unique pupil pattern
    for j in range(n_pupil):
        sensitivity[:, j, :] += j  # Add pupil index
    
    print("=== Labeled Sensitivity Matrix Test ===")
    print(f"Each layer k has base value k*100, plus pupil index j")
    print(f"\nLayer 0, pupil 0: {sensitivity[0, 0, 0]}")
    print(f"Layer 0, pupil 5: {sensitivity[0, 5, 0]}")
    print(f"Layer 2, pupil 0: {sensitivity[2, 0, 0]}")
    print(f"Layer 2, pupil 5: {sensitivity[2, 5, 0]}")
    
    A = sensitivity.reshape(-1, n_dof)
    
    # Check mapping
    print(f"\nAfter reshape to ({A.shape[0]}, {A.shape[1]}):")
    print(f"A[0, 0] (should be 0*100 + 0 = 0): {A[0, 0]}")
    print(f"A[5, 0] (should be 0*100 + 5 = 5): {A[5, 0]}")
    print(f"A[{2*n_pupil}, 0] (should be 2*100 + 0 = 200): {A[2*n_pupil, 0]}")
    print(f"A[{2*n_pupil + 5}, 0] (should be 2*100 + 5 = 205): {A[2*n_pupil + 5, 0]}")


# Run tests
test_array, A = test_reshape_indexing()
print("\n" + "="*60 + "\n")
test_with_labeled_sensitivity()

In [ ]:
def plot_sensitivity_matrix_layer(sensitivity_layer, pupil_indices, k_index, 
                                   output_path):
    """Plot one focal Zernike layer of the sensitivity matrix as a heatmap.
    
    Parameters
    ----------
    sensitivity_layer : array_like, shape (n_pupil, n_dof)
        One layer (focal Zernike k) of the sensitivity matrix
    pupil_indices : list of int
        Pupil Zernike indices (e.g., [4,5,...,19,22,...,26])
    k_index : int
        The focal Zernike index for this layer
    output_path : Path
        Where to save the figure
    vmin, vmax : float, optional
        Color scale limits
    """
    sensitivity_layer = np.asarray(sensitivity_layer)
    n_pupil, n_dof = sensitivity_layer.shape
    
    # Split into subsystems
    # Assuming DOF order: [M2_hex(5), Cam_hex(5), M1M3_bends(20), M2_bends(20)]
    m2_xyz = sensitivity_layer[:, 0:3]      # z, x, y, rx, ry
    m2_rxy = sensitivity_layer[:, 3:5] / 3600.    # z, x, y, rx, ry
    cam_xyz = sensitivity_layer[:, 5:8]    # z, x, y, rx, ry
    cam_rxy = sensitivity_layer[:, 8:10] / 3600.    # z, x, y, rx, ry
    m1m3_bends = sensitivity_layer[:, 10:30]
    m2_bends = sensitivity_layer[:, 30:50]
    
    # Create figure with two subplots
    fig, (ax_hex, ax_bends) = plt.subplots(1, 2, figsize=(16, 6),
                                            gridspec_kw={'width_ratios': [1, 3]})
    
    # Plot hexapods
    hexapod_data = np.concatenate([m2_xyz, m2_rxy, cam_xyz, cam_rxy], axis=1)

    # Determine color scale if not provided
    abs_max = np.max(np.abs(hexapod_data))
    vmin = -abs_max
    vmax = abs_max

    im_hex = ax_hex.imshow(hexapod_data, aspect='auto', cmap='RdBu_r',
                           vmin=vmin, vmax=vmax, interpolation='nearest')
    
    # Hexapod labels
    hex_labels = ['z', 'x', 'y', r'$r_x$', r'$r_y$', 
                  'z', 'x', 'y', r'$r_x$', r'$r_y$']
    ax_hex.set_xticks(range(10))
    ax_hex.set_xticklabels(hex_labels, fontsize=10)
    ax_hex.set_yticks(range(n_pupil))
    ax_hex.set_yticklabels([f'$Z_{{{j}}}$' for j in pupil_indices], fontsize=9)
    ax_hex.set_xlabel('M2          Camera', fontsize=11)
    
    # Add vertical line to separate M2 and Camera
    ax_hex.axvline(4.5, color='gray', lw=1.5, ls='--')
    
    # Plot bending modes
    bends_data = np.concatenate([m1m3_bends, m2_bends], axis=1)

    # Determine color scale if not provided
    abs_max = np.max(np.abs(bends_data))
    vmin = -abs_max
    vmax = abs_max
 
    im_bends = ax_bends.imshow(bends_data, aspect='auto', cmap='RdBu_r',
                               vmin=vmin, vmax=vmax, interpolation='nearest')
    
    # Bending mode labels
    bend_labels = ([f'$B_{{1,{i}}}$' for i in range(1, 21)] + 
                   [f'$B_{{2,{i}}}$' for i in range(1, 21)])
    # Show every 5th label to avoid crowding
    tick_positions = list(range(0, 40, 5))
    ax_bends.set_xticks(tick_positions)
    ax_bends.set_xticklabels([bend_labels[i] for i in tick_positions], fontsize=10)
    ax_bends.set_yticks(range(n_pupil))
    ax_bends.set_yticklabels([f'$Z_{{{j}}}$' for j in pupil_indices], fontsize=9)
    ax_bends.set_xlabel('M1M3                                        M2', fontsize=11)
    
    # Add vertical line to separate M1M3 and M2
    ax_bends.axvline(19.5, color='gray', lw=1.5, ls='--')
    
    # Colorbars
    cbar_hex = plt.colorbar(im_hex, ax=ax_hex, fraction=0.046, pad=0.04)
    cbar_hex.set_label(r'$\mu m$ or $\mu m$/arcsec', fontsize=10)
    
    cbar_bends = plt.colorbar(im_bends, ax=ax_bends, fraction=0.046, pad=0.04)
    cbar_bends.set_label(r'$\mu m$ or $\mu m$/arcsec', fontsize=10)
    
    # Title
    fig.suptitle(f'Sensitivity Matrix for Focal Zernike $Z_{{{k_index}}}$', 
                 fontsize=13, y=0.98)
    
    plt.tight_layout()
    print(f"  Saving sensitivity matrix plot to {output_path}")
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


# Usage example:
def plot_all_sensitivity_layers(sensitivity_matrix, pupil_indices, n_focal, output_dir):
    """Plot all focal Zernike layers of sensitivity matrix.
    
    Parameters
    ----------
    sensitivity_matrix : array_like, shape (n_focal, n_pupil, n_dof)
    pupil_indices : list of int
    n_focal : int
    output_dir : Path
    """
    
    # # Determine global color scale across all layers
    # abs_max = np.max(np.abs(sensitivity_matrix))
    
    for k in range(1, n_focal):
        output_path = output_dir / f'sensitivity_k{k}.png'
        plot_sensitivity_matrix_layer(sensitivity_matrix[k], pupil_indices, 
                                       k, output_path)

In [ ]:
def plot_all_sensitivity_layers(sensitivity_matrix, pupil_indices, n_focal, output_path):
    """Plot all focal Zernike layers of sensitivity matrix in one figure.
    
    Parameters
    ----------
    sensitivity_matrix : array_like, shape (n_focal, n_pupil, n_dof)
    pupil_indices : list of int
        Pupil Zernike indices (e.g., [4,5,...,19,22,...,26])
    n_focal : int
        Number of focal Zernikes to plot
    output_path : Path
        Where to save the figure
    """
    sensitivity_matrix = np.asarray(sensitivity_matrix)
    n_pupil = len(pupil_indices)
    
    # Create figure with subplots: n_focal rows, 2 columns (hexapod, bends)
    fig = plt.figure(figsize=(16, 4 * n_focal))
    gs = fig.add_gridspec(n_focal, 2, hspace=0.3, wspace=0.2, 
                          width_ratios=[1, 3])
    
    for k, kk in enumerate(range(1, n_focal+1)):
        sensitivity_layer = sensitivity_matrix[kk]
        
        # Split into subsystems
        # Assuming DOF order: [M2_hex(5), Cam_hex(5), M1M3_bends(20), M2_bends(20)]
        m2_xyz = sensitivity_layer[:, 0:3]
        m2_rxy = sensitivity_layer[:, 3:5] / 3600.
        cam_xyz = sensitivity_layer[:, 5:8]
        cam_rxy = sensitivity_layer[:, 8:10] / 3600.
        m1m3_bends = sensitivity_layer[:, 10:30]
        m2_bends = sensitivity_layer[:, 30:50]
        
        # Create subplots for this row
        ax_hex = fig.add_subplot(gs[k, 0])
        ax_bends = fig.add_subplot(gs[k, 1])
        
        # Plot hexapods
        hexapod_data = np.concatenate([m2_xyz, m2_rxy, cam_xyz, cam_rxy], axis=1)
        abs_max_hex = np.max(np.abs(hexapod_data))
        
        im_hex = ax_hex.imshow(hexapod_data, aspect='auto', cmap='RdBu_r',
                               vmin=-abs_max_hex, vmax=abs_max_hex, 
                               interpolation='nearest')
        
        # Hexapod labels
        hex_labels = ['z', 'x', 'y', r'$r_x$', r'$r_y$', 
                      'z', 'x', 'y', r'$r_x$', r'$r_y$']
        ax_hex.set_xticks(range(10))
        ax_hex.set_xticklabels(hex_labels, fontsize=10)
        ax_hex.set_yticks(range(n_pupil))
        ax_hex.set_yticklabels([f'$Z_{{{j}}}$' for j in pupil_indices], fontsize=9)
        
        # Only label x-axis on bottom row
        if k == n_focal - 1:
            ax_hex.set_xlabel('M2          Camera', fontsize=11)
        
        # Add row label (focal Zernike index)
        ax_hex.set_ylabel(f'$Z_{{{kk}}}$', fontsize=12, fontweight='bold')
        
        # Add vertical line to separate M2 and Camera
        ax_hex.axvline(4.5, color='gray', lw=1.5, ls='--')
        
        # Plot bending modes
        bends_data = np.concatenate([m1m3_bends, m2_bends], axis=1)
        abs_max_bends = np.max(np.abs(bends_data))
        
        im_bends = ax_bends.imshow(bends_data, aspect='auto', cmap='RdBu_r',
                                   vmin=-abs_max_bends, vmax=abs_max_bends, 
                                   interpolation='nearest')
        
        # Bending mode labels
        bend_labels = ([f'$B_{{1,{i}}}$' for i in range(1, 21)] + 
                       [f'$B_{{2,{i}}}$' for i in range(1, 21)])
        tick_positions = list(range(0, 40, 5))
        ax_bends.set_xticks(tick_positions)
        ax_bends.set_xticklabels([bend_labels[i] for i in tick_positions], fontsize=10)
        ax_bends.set_yticks(range(n_pupil))
        ax_bends.set_yticklabels([f'$Z_{{{j}}}$' for j in pupil_indices], fontsize=9)
        
        # Only label x-axis on bottom row
        if k == n_focal - 1:
            ax_bends.set_xlabel('M1M3                                        M2', fontsize=11)
        
        # Add vertical line to separate M1M3 and M2
        ax_bends.axvline(19.5, color='gray', lw=1.5, ls='--')
        
        # Colorbars
        cbar_hex = plt.colorbar(im_hex, ax=ax_hex, fraction=0.046, pad=0.04)
        cbar_hex.set_label(r'$\mu m/\mu m$ or $\mu m$/arcsec', fontsize=9)
        
        cbar_bends = plt.colorbar(im_bends, ax=ax_bends, fraction=0.046, pad=0.04)
        cbar_bends.set_label(r'$\mu m/\mu m$ or $\mu m$/arcsec', fontsize=9)
    
    # Overall title
    fig.suptitle('Sensitivity Matrix (all focal Zernikes)', fontsize=14)
    
    print(f"  Saving sensitivity matrix plot to {output_path}")
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

In [ ]:
# Example call:
pupil_indices_plot = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26]
# plot_all_sensitivity_layers(ideal_sens_dz.coef, pupil_indices, n_focal, output_dir)

# Usage:
plot_all_sensitivity_layers(ideal_sens_dz.coef, pupil_indices_plot, n_focal, 
                            output_dir / f'sensitivity_k1-{n_focal}.pdf')

# Analysis for Aaron's DZ values

In [ ]:
bin2_dir = '/home/r/roodman/u/LSST/notebooks/rubin-work/aos/output/fam_danish_OCS_wep_v16_8_0_dviz_v3_5_0_20260315_20260317/'
fn = 'intrinsic_fits_wep_v16_8_0_dviz_v3_5_0_20260315_20260317.parquet'
dz_tab = QTable.read(bin2_dir + fn)

In [ ]:
dz_tab

In [ ]:
dz_tab.colnames

In [ ]:
dz_tab['z1toz6_bad_fit'] == 0.

In [ ]:
dz_tab = dz_tab[dz_tab['z1toz6_bad_fit'] == 0.]

In [ ]:
dz_tab

In [ ]:
# extract the matrix of DZ values to multiply with the sensitivity matrix
# have to get rid of final 'bad_fit' col
z1toz6_cols = [col for col in dz_daily_median.colnames if col.startswith('z1toz6') and 'err' not in col and 'scale' not in col and 'bad' not in col]

## Plot rotator angle info

In [ ]:
fig, ax = plt.subplots(1,1)
ax.scatter(dz_tab['day_obs'], dz_tab['rotAngle'])

## Grouping functions

In [ ]:
def group_by_tolerance(values, tolerance=1.):
    """
    Group values that are close within tolerance.
    
    Returns a list of lists, where each sublist contains indices
    of values that are close to each other.
    """
    values = np.asarray(values)
    used = np.zeros(len(values), dtype=bool)
    groups = []
    
    for i in range(len(values)):
        if used[i]:
            continue
        # Find all values close to this one
        close_mask = np.isclose(values, values[i], atol=tolerance, rtol=0)
        group_indices = np.where(close_mask)[0]
        groups.append(group_indices.tolist())
        used[close_mask] = True
    
    return groups

def assign_groups(values, tolerance=1.):
    """Return an array of group labels for each value."""
    groups = group_by_tolerance(values, tolerance)
    labels = np.zeros(len(values), dtype=int)
    for group_id, indices in enumerate(groups):
        labels[indices] = group_id
    return labels

In [ ]:
def median_per_group(group_idx_list, n_focal, n_pupil):
    """Given a list of grouped indices, returns the set of DZs median'ed over the entries in each group.
    Note z1toz6 cols are currently hardcoded in!
    """
    z1toz6_cols = [col for col in dz_tab.colnames if col.startswith('z1toz6')
                   and 'err' not in col and 'scale' not in col and 'bad' not in col]

    grp_med_dz_arr_list = []
    for group_idxs in group_idx_list:
        median_row = [np.median(dz_tab[col][group_idxs]) for col in z1toz6_cols]
        grp_med_dz_arr_list.append(np.array(median_row).reshape(n_pupil, n_focal).T)

    return grp_med_dz_arr_list

In [ ]:
rot_groups = group_by_tolerance(dz_tab['rotAngle'])
rotang_order_idx = [2,4,0,3,1]
rot_groups_ordered = [rot_groups[i] for i in rotang_order_idx]

In [ ]:
len(rot_groups)

In [ ]:
fig, ax = plt.subplots(1,1)
for group in rot_groups_ordered:
    ax.scatter(dz_tab['day_obs'][group], dz_tab['rotAngle'][group])

In [ ]:
fig, ax = plt.subplots(1,1)
for group in rot_groups_ordered:
    ax.scatter(dz_tab['image_idx'][group], dz_tab['rotAngle'][group])

### Debugging indices

In [ ]:
print(focal_indices)
print(pupil_indices)

In [ ]:
nrow, ncol=3, 7
for i, k in enumerate(focal_indices):
    fig, axs = plt.subplots(nrow, ncol, figsize=(30,10))
    fig.suptitle(f"k={k}", fontsize=15)
    for group in rot_groups_ordered:
        for l, j in enumerate(pupil_indices):
            ax = axs[l//ncol,l%ncol]
            ax.scatter(dz_tab['image_idx'][group], dz_tab[f"z1toz6_z{j}_c{k}"][group])
            ax.set_title(j, fontsize=12)

In [ ]:
nrow, ncol=3, 7
n_focal, n_pupil = 6, 21
med_dz_grp_list = median_per_group(rot_groups_ordered, n_focal, n_pupil)
for i, k in enumerate(focal_indices):
    fig, axs = plt.subplots(nrow, ncol, figsize=(30,10))
    fig.suptitle(f"k={k}", fontsize=15)
    for group_idxs, group_dz_med in zip(rot_groups_ordered, med_dz_grp_list):
        for l, j in enumerate(pupil_indices):
            ax = axs[l//ncol,l%ncol]
            ax.scatter(np.median(dz_tab['rotAngle'][group_idxs]), group_dz_med[i,l])
            ax.set_title(j, fontsize=12)

In [ ]:
# checking medians
rot_m60 = med_dz_grp_list[0]
print(rot_m60[4,4])
print(rot_m60[5,5])
print(rot_m60[4,:])
np.median(dz_tab['z1toz6_z5_c5'][rot_groups_ordered[0]])

In [ ]:
med_dz_grp_list_2 = median_per_group(rot_groups_ordered, n_focal, n_pupil)
for rot_dz in med_dz_grp_list_2:
    print(rot_dz[4,4])
np.median(dz_tab['z1toz6_z5_c5'][rot_groups_ordered[0]])

In [ ]:
np.array([np.median(dz_tab[col][rot_groups_ordered[0]]) for col in z1toz6_cols]).reshape(n_focal, n_pupil)[4,4]

In [ ]:
np.array([np.median(np.array(dz_tab[col])[rot_groups_ordered[0]]) for col in z1toz6_cols]).reshape(n_focal, n_pupil)[4,4]

In [ ]:
grouped_tab = dz_tab[rot_groups_ordered[0]]
np.array([np.median(grouped_tab[col]) for col in z1toz6_cols]).reshape(n_pupil, n_focal)[1,4]

In [ ]:
print(np.array(z1toz6_cols).reshape(n_pupil, n_focal)[1,4])
print(np.array(z1toz6_cols).reshape(n_focal, n_pupil)[4,1])

In [ ]:
np.array(z1toz6_cols).reshape(n_pupil, n_focal)[1,4]

## Split by day

In [ ]:
dz_tab_day_split = [dz_tab[dz_tab['day_obs'] == do_val] for do_val in day_obs_vals]

In [ ]:
dz_tab_day_split

In [ ]:
dz_median_per_day = []
for tab in dz_tab_day_split:
    median_row = QTable({col: [np.median(tab[col])] for col in tab.colnames if col.startswith('z')})
    dz_median_per_day.append(median_row)

In [ ]:
dz_median_per_day

In [ ]:
dz_daily_median = vstack(dz_median_per_day)

In [ ]:
dz_daily_median

In [ ]:
dz_daily_median.add_column(day_obs_vals, name='day_obs', index=0)

In [ ]:
dz_daily_median

In [ ]:
# extract the matrix of DZ values to multiply with the sensitivity matrix
# have to get rid of final 'bad_fit' col
z1toz6_cols = [col for col in dz_daily_median.colnames if col.startswith('z1toz6') and 'err' not in col and 'scale' not in col and 'bad' not in col]

In [ ]:
z1toz6_cols

In [ ]:
dz_data_daily = [np.array([dz_daily_median[i][col] for col in z1toz6_cols]).reshape(6, 21) for i in range(3)]

In [ ]:
for i in range(3):
    print(i)
    print(dz_data_daily[i].shape, dz_data_daily[i])

In [ ]:
pupil_indices = np.concatenate([np.arange(4,20), np.arange(22,27)])

In [ ]:
print(pupil_indices, pupil_indices.shape)

In [ ]:
focal_indices = np.arange(1,7)

In [ ]:
focal_indices

In [ ]:
ideal_sens_dz.coef[np.ix_(focal_indices, pupil_indices)][..., :]

In [ ]:
model_smatrix = ideal_sens_dz.coef[np.ix_(focal_indices, pupil_indices)][..., :]

In [ ]:
model_smatrix.shape

### Solve for DOFs

In [ ]:
# Flatten
y = dz_data_daily[0].reshape(-1)
A = model_smatrix.reshape(-1, 50)

# Solve
x_hat, residuals, rank, s = np.linalg.lstsq(A, y, rcond=1e-3)

print("Recovered DOFs shape:", x_hat.shape)
print("Rank:", rank)
print("Singular values:", s)

In [ ]:
for i, (date, dz_data) in enumerate(zip(dz_daily_median['day_obs'], dz_data_daily)):
    y = dz_data.reshape(-1) 
    print(y / dz_data_daily[0].reshape(-1))

### Print

In [ ]:
def print_dofs(x_hat):

    dof_labels = [ "M2_hex_z", "M2_hex_x", "M2_hex_y", "M2_hex_rx", "M2_hex_ry", "Cam_hex_z", "Cam_hex_x", "Cam_hex_y", "Cam_hex_rx", "Cam_hex_ry", ] 
    dof_labels += [f"M1M3_B{i}" for i in range(1, 21)] 
    dof_labels += [f"M2_B{i}" for i in range(1, 21)]
    
    x_hat = np.asarray(x_hat)
    label_to_value = dict(zip(dof_labels, x_hat))
    
    left_groups = [
        ("Camera hexapod", dof_labels[5:10]),
        ("M1M3 bends", dof_labels[10:30]),
    ]
    
    right_groups = [
        ("M2 hexapod", dof_labels[:5]),
        ("M2 bends", dof_labels[30:50]),
    ]

    left_data = build_rows(left_groups, label_to_value)
    right_data = build_rows(right_groups, label_to_value)
    
    all_rows = [row for _, rows in (left_data + right_data) for row in rows]
    label_w = max(len("DOF"), max(len(r[0]) for r in all_rows))
    value_w = max(len("Value"), 14)
    unit_w = max(len("Unit"), max(len(r[2]) for r in all_rows))

    left_lines = flatten_blocks(left_data)
    right_lines = flatten_blocks(right_data)
    
    left_width = max(len(line) for line in left_lines)
    right_width = max(len(line) for line in right_lines)
    n_lines = max(len(left_lines), len(right_lines))
    
    left_lines += [""] * (n_lines - len(left_lines))
    right_lines += [""] * (n_lines - len(right_lines))
    
    gap = "   "
    
    for l, r in zip(left_lines, right_lines):
        print(f"{l:<{left_width}}{gap}{r}")


def build_rows(group_specs, label_to_value):
    out = []
    for group_name, labels in group_specs:
        rows = []
        for label in labels:
            val = label_to_value[label]
            if label.endswith("_rx") or label.endswith("_ry"):
                display_val = val * 3600.0
                unit = "arcsec"
            else:
                display_val = val
                unit = "um"
            rows.append((label, display_val, unit))
        out.append((group_name, rows))
    return out


def make_block(group_name, rows):
    line = f"+-{'-'*label_w}-+-{'-'*value_w}-+-{'-'*unit_w}-+"
    group_line_w = len(line) - 4

    block = []
    block.append(line)
    block.append(f"| {'DOF':<{label_w}} | {'Value':>{value_w}} | {'Unit':<{unit_w}} |")
    block.append(line)
    block.append(f"| {(' ' + group_name + ' '):<{group_line_w}} |")
    block.append(line)
    for label, value, unit in rows:
        block.append(f"| {label:<{label_w}} | {value:>{value_w}.6f} | {unit:<{unit_w}} |")
    block.append(line)
    return block

def flatten_blocks(grouped_blocks):
    lines = []
    for i, (group_name, rows) in enumerate(grouped_blocks):
        lines.extend(make_block(group_name, rows))
        if i != len(grouped_blocks) - 1:
            lines.append("")
    return lines

In [ ]:
A = model_smatrix.reshape(-1, 50)
print(A)

for i, (date, dz_data) in enumerate(zip(dz_daily_median['day_obs'], dz_data_daily)):

    y = dz_data.reshape(-1)
    x_hat, residuals, rank, s = np.linalg.lstsq(A, y, rcond=1e-3)

    print(date)
    print_dofs(x_hat)
    print()

In [ ]:
A = model_smatrix.reshape(-1, 50)
print(A.shape)

for i, (date, dz_data) in enumerate(zip(dz_daily_median['day_obs'], dz_data_daily)):

    y = dz_data.reshape(-1)
    x_hat, residuals, rank, s = np.linalg.lstsq(A, y, rcond=1e-3)

    y_reconstructed = A @ x_hat
    residual_vector = y - y_reconstructed
    residual_sum_of_squares = np.sum(residual_vector**2)
    rms_residual = np.sqrt(np.mean(residual_vector**2))

    print(f"Residual sum of squares: {residual_sum_of_squares}")
    print(f"RMS residual: {rms_residual}")

In [ ]:
def print_residuals(residual_vector, focal_indices, pupil_indices, tolerance=0.01):
    """
    Print residuals in a 2D table format (transposed: pupil as rows, focal as columns).
    
    Parameters
    ----------
    residual_vector : array_like
        Flattened residual vector of length (n_focal * n_pupil)
    focal_indices : list
        Column labels (focal plane indices)
    pupil_indices : list
        Row labels (pupil plane indices)
    tolerance : float
        Minimum absolute value to mark with asterisk
    """
    residuals_2d = residual_vector.reshape(len(focal_indices), len(pupil_indices))
    residuals_2d = residuals_2d.T  # Transpose
    
    # Determine column widths (add 2 for space + potential asterisk)
    pupil_label_w = max(len(str(idx)) for idx in pupil_indices)
    pupil_label_w = max(pupil_label_w, len("Pupil"))
    
    focal_label_w = max(len(str(idx)) for idx in focal_indices)
    focal_label_w = max(focal_label_w, 12)  # At least wide enough for values + space + asterisk
    
    # Header
    header = f"{'j \ k':<{pupil_label_w}}"
    for focal_idx in focal_indices:
        header += f" | {str(focal_idx):>{focal_label_w}}"
    print(header)
    print("-" * len(header))
    
    # Data rows
    for i, pupil_idx in enumerate(pupil_indices):
        row = f"{str(pupil_idx):<{pupil_label_w}}"
        for j, focal_idx in enumerate(focal_indices):
            val = residuals_2d[i, j]
            marker = "*" if abs(val) > tolerance else " "
            row += f" | {val:>{focal_label_w-2}.6f} {marker}"
        print(row)

In [ ]:
for i, (date, dz_data) in enumerate(zip(dz_daily_median['day_obs'], dz_data_daily)):

    y = dz_data.reshape(-1)
    x_hat, residuals, rank, s = np.linalg.lstsq(A, y, rcond=1e-3)

    y_reconstructed = A @ x_hat
    residual_vector = y - y_reconstructed
    residual_sum_of_squares = np.sum(residual_vector**2)
    rms_residual = np.sqrt(np.mean(residual_vector**2))

    print(date)
    print_residuals(residual_vector, focal_indices, pupil_indices, tolerance=0.05)
    print(f"Residual sum of squares: {residual_sum_of_squares}")
    print(f"RMS residual: {rms_residual}\n")

## Split by rotator angle

In [ ]:
rot_groups_unordered = group_by_tolerance(dz_tab['rotAngle'])
rotang_order_idx = [2,4,0,3,1]
rot_groups = [rot_groups_unordered[i] for i in rotang_order_idx]

In [ ]:
len(rot_groups)

In [ ]:
rotang_labels = []
for i, group in enumerate(rot_groups):
    rotangle = np.round(np.mean(dz_tab['rotAngle'][group]))
    rotang_labels.append(rotangle)

In [ ]:
print(rotang_labels)

In [ ]:
fig, ax = plt.subplots(1,1)
for i, group in enumerate(rot_groups):
    rotangle = np.round(np.mean(dz_tab['rotAngle'][group]))
    ax.scatter(dz_tab['day_obs'][group], dz_tab['rotAngle'][group], label=rotangle)
ax.legend(loc=(1.01, 0.))

In [ ]:
A = model_smatrix.reshape(-1, 50)                  # (368, 50)
for i, (rotang, dz_data) in enumerate(zip(rotang_labels, dz_arr_list)):

    y = dz_data.reshape(-1)
    x_hat, residuals, rank, s = np.linalg.lstsq(A, y, rcond=1e-3)

    print(100*"-")
    print(rotang)
    print_dofs(x_hat)
    print()

    y_reconstructed = A @ x_hat
    residual_vector = y - y_reconstructed
    residual_sum_of_squares = np.sum(residual_vector**2)
    rms_residual = np.sqrt(np.mean(residual_vector**2))

    print_residuals(residual_vector, focal_indices, pupil_indices, tolerance=0.05)
    print(f"Residual sum of squares: {residual_sum_of_squares}")
    print(f"RMS residual: {rms_residual}\n")

## Plot DZ and DOF coefficients

### Plotting functions

In [ ]:
def setup_dz_figure(n_focal_zernikes, pupil_indices, n_datasets):
    """Create a figure with one subplot per focal Zernike k, pupil j on the x-axis.

    Parameters
    ----------
    n_focal_zernikes : int
        Number of focal Zernikes (k indices, typically 6 for k=0-5)
    pupil_indices : list of int
        Actual pupil Zernike indices (e.g., [4,5,6,...,19,22,23,...,26])
    n_datasets : int
        Number of datasets to plot

    Returns
    -------
    fig : Figure
    axes : list of Axes
    dataset_width : float
        Stagger offset between datasets at a given j position.
    pupil_positions : dict
        Maps pupil index j to x-axis position
    """
    n_subplots = n_focal_zernikes

    n_pupil = len(pupil_indices)
    fig_width = max(10, n_pupil * 0.4 + n_datasets * 0.3) + 4
    fig_height = max(4, n_subplots * 2.2)

    fig, axes = plt.subplots(n_subplots, 1, figsize=(fig_width, fig_height),
                             sharex=True, constrained_layout=True)
    if n_subplots == 1:
        axes = [axes]

    stagger_total = 0.95
    dataset_width = stagger_total / n_datasets

    # Create mapping from pupil index to x position
    pupil_positions = {j: i for i, j in enumerate(pupil_indices)}

    # Alternating shading for even j
    for i, j in enumerate(pupil_indices):
        if j % 2 == 0:
            for ax in axes:
                ax.axvspan(i - 0.5, i + 0.5, color='k', alpha=0.07, lw=0)

    # Per-subplot formatting
    for k, ax in enumerate(axes):
        ax.set_ylabel(f"k={k+1}", fontsize=11)
        ax.axhline(0, color='gray', lw=0.5, ls='-')
        ax.grid(True, axis='y', alpha=0.4)

    # X-axis on bottom subplot
    axes[-1].set_xticks(range(len(pupil_indices)))
    axes[-1].set_xticklabels([str(j) for j in pupil_indices], fontsize=9)
    axes[-1].set_xlim(-0.5, len(pupil_indices) - 0.5)
    axes[-1].set_xlabel("Pupil Zernike Index (j)", fontsize=11)

    return fig, axes, dataset_width, pupil_positions


def plot_dz_matrix(axes, dz_matrix, pupil_positions, dataset_idx, 
                   dataset_width, color, marker='o'):
    """Plot DZ matrix coefficients on k-j subplots.
    
    Parameters
    ----------
    axes : list of Axes
        One axis per focal Zernike k
    dz_matrix : array_like, shape (n_focal, n_pupil)
        Double Zernike coefficient matrix
    pupil_positions : dict
        Maps pupil index j to x-axis position
    dataset_idx : int
        Which dataset (for staggering)
    dataset_width : float
        Stagger amount
    color : color
    marker : str
    """
    dz_matrix = np.asarray(dz_matrix)
    n_focal, n_pupil = dz_matrix.shape
    
    x_offset = (dataset_idx - 0.5) * dataset_width - 0.25
    
    for k in range(n_focal):
        # Plot all pupil indices for this focal index
        x_positions = np.arange(n_pupil) + x_offset
        y_values = dz_matrix[k, :]
        
        axes[k].plot(x_positions, y_values, marker=marker, color=color,
                    linestyle='none', markersize=6)


def finalize_dz_figure(fig, axes, file_keys, dataset_colors,
                       title, output_path, marker_size=6):
    """Add legend, title, and save a DZ k-j subplot figure.

    Parameters
    ----------
    fig : Figure
    axes : list of Axes
    file_keys : list of str
        One label per dataset
    dataset_colors : sequence of colors
    title : str
        Figure suptitle text.
    output_path : Path
        Full path to save the figure.
    marker_size : float
    """
    # Legend on bottom subplot
    handles, labels = [], []
    for file_idx, file_key in enumerate(file_keys):
        color = dataset_colors[file_idx]
        marker = 'o'
        handles.append(plt.Line2D([0], [0], color=color, marker=marker,
                                  linestyle='none', markersize=marker_size))
        labels.append(file_key)
    axes[-1].legend(handles, labels, ncols=1, loc='lower left',
                    bbox_to_anchor=(1.01, 0.), fontsize=11)

    fig.suptitle(title, fontsize=12)

    print(f"  Saving DZ plot to {output_path}")
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


# Usage example:
def plot_dz_datasets(dz_matrix_list, pupil_indices, file_keys, 
                     dataset_colors, title, output_path):
    """Plot multiple DZ coefficient matrices.
    
    Parameters
    ----------
    dz_matrix_list : list of array_like
        Each array is shape (n_focal, n_pupil), e.g., (6, 21)
    pupil_indices : list of int
        Actual pupil Zernike indices, e.g., [4,5,6,...,19,22,23,...,26]
    file_keys : list of str
    dataset_colors : list of colors
    title : str
    output_path : Path
    """
    n_datasets = len(dz_matrix_list)
    n_focal = dz_matrix_list[0].shape[0]
    
    fig, axes, dataset_width, pupil_positions = setup_dz_figure(
        n_focal, pupil_indices, n_datasets)
    
    for dataset_idx, (dz_matrix, color) in enumerate(zip(dz_matrix_list, dataset_colors)):
        marker = 'o'
        
        plot_dz_matrix(axes, dz_matrix, pupil_positions, dataset_idx,
                      dataset_width, color, marker)
    
    finalize_dz_figure(fig, axes, file_keys, dataset_colors, 
                      title, output_path)

In [ ]:
def setup_dof_figure(n_datasets):
    """Create a figure with one subplot per DOF group.
    
    Returns
    -------
    fig : Figure
    axes : dict
        Keys: 'xyz', 'rxry', 'm1m3_bends', 'm2_bends'
    dataset_width : float
        Stagger offset between datasets.
    """
    fig = plt.figure(figsize=(14, 12))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    
    axes = {
        'xyz': fig.add_subplot(gs[0, 0]),
        'rxry': fig.add_subplot(gs[0, 1]),
        'm1m3_bends': fig.add_subplot(gs[1, :]),  # Full width
        'm2_bends': fig.add_subplot(gs[2, :]),    # Full width
    }
    
    stagger_total = 0.8
    dataset_width = stagger_total / n_datasets
    
    # XYZ (Cam and M2 hexapod translations)
    xyz_labels = ['Cam_z', 'Cam_x', 'Cam_y', 'M2_z', 'M2_x', 'M2_y']
    axes['xyz'].set_xticks(range(len(xyz_labels)))
    axes['xyz'].set_xticklabels(xyz_labels, rotation=45, ha='right')
    axes['xyz'].set_ylabel('Value (μm)', fontsize=10)
    axes['xyz'].set_title('Hexapod Translations', fontsize=11)
    axes['xyz'].axhline(0, color='gray', lw=0.5, ls='-')
    axes['xyz'].grid(True, axis='y', alpha=0.4)
    # Alternating shading
    for i in range(0, len(xyz_labels), 2):
        axes['xyz'].axvspan(i - 0.5, i + 0.5, color='k', alpha=0.07, lw=0)
    
    # RxRy (Cam and M2 hexapod rotations)
    rxry_labels = ['Cam_rx', 'Cam_ry', 'M2_rx', 'M2_ry']
    axes['rxry'].set_xticks(range(len(rxry_labels)))
    axes['rxry'].set_xticklabels(rxry_labels, rotation=45, ha='right')
    axes['rxry'].set_ylabel('Value (arcsec)', fontsize=10)
    axes['rxry'].set_title('Hexapod Rotations', fontsize=11)
    axes['rxry'].axhline(0, color='gray', lw=0.5, ls='-')
    axes['rxry'].grid(True, axis='y', alpha=0.4)
    # Alternating shading
    for i in range(0, len(rxry_labels), 2):
        axes['rxry'].axvspan(i - 0.5, i + 0.5, color='k', alpha=0.07, lw=0)
    
    # M1M3 bending modes
    axes['m1m3_bends'].set_xticks(range(20))
    axes['m1m3_bends'].set_xticklabels([str(i+1) for i in range(20)])
    axes['m1m3_bends'].set_ylabel('Value (μm)', fontsize=10)
    axes['m1m3_bends'].set_title('M1M3 Bending Modes', fontsize=11)
    axes['m1m3_bends'].axhline(0, color='gray', lw=0.5, ls='-')
    axes['m1m3_bends'].grid(True, axis='y', alpha=0.4)
    # Alternating shading
    for i in range(0, 20, 2):
        axes['m1m3_bends'].axvspan(i - 0.5, i + 0.5, color='k', alpha=0.07, lw=0)
    
    # M2 bending modes
    axes['m2_bends'].set_xticks(range(20))
    axes['m2_bends'].set_xticklabels([str(i+1) for i in range(20)])
    axes['m2_bends'].set_xlabel('Bending Mode', fontsize=10)
    axes['m2_bends'].set_ylabel('Value (μm)', fontsize=10)
    axes['m2_bends'].set_title('M2 Bending Modes', fontsize=11)
    axes['m2_bends'].axhline(0, color='gray', lw=0.5, ls='-')
    axes['m2_bends'].grid(True, axis='y', alpha=0.4)
    # Alternating shading
    for i in range(0, 20, 2):
        axes['m2_bends'].axvspan(i - 0.5, i + 0.5, color='k', alpha=0.07, lw=0)
    
    return fig, axes, dataset_width


def plot_dof_vector(ax, x_positions, values, dataset_idx, dataset_width, 
                    color, marker='o'):
    """Plot DOF values at staggered x positions.
    
    Parameters
    ----------
    ax : Axes
    x_positions : array_like
        Base x positions
    values : array_like
        DOF values to plot
    dataset_idx : int
        Which dataset (for staggering)
    dataset_width : float
        Stagger amount
    color : color
    marker : str
    """
    x_offset = (dataset_idx - 0.5) * dataset_width - 0.25
    x_plot = np.array(x_positions) + x_offset
    
    ax.plot(x_plot, values, marker=marker, color=color, 
            linestyle='none', markersize=6)


def plot_dof_datasets(x_hat_list, file_keys, dataset_colors, title, output_path):
    """Plot multiple DOF solution vectors.
    
    Parameters
    ----------
    x_hat_list : list of array_like
        Each array is a 50-element DOF solution vector
        Order: [M2_hex(5), Cam_hex(5), M1M3_bends(20), M2_bends(20)]
    file_keys : list of str
    dataset_colors : list of colors
    title : str
    output_path : Path
    """
    n_datasets = len(x_hat_list)
    fig, axes, dataset_width = setup_dof_figure(n_datasets)
    
    for dataset_idx, (x_hat, color) in enumerate(zip(x_hat_list, dataset_colors)):
        marker = 'o'
        
        # Split x_hat into subsystems
        # [0:5] M2 hexapod: [z, x, y, rx, ry]
        # [5:10] Cam hexapod: [z, x, y, rx, ry]
        # [10:30] M1M3 bends
        # [30:50] M2 bends
        
        m2_hex = x_hat[0:5]
        cam_hex = x_hat[5:10]
        m1m3_bends = x_hat[10:30]
        m2_bends = x_hat[30:50]
        
        # XYZ: Cam z,x,y then M2 z,x,y
        xyz_values = np.concatenate([cam_hex[0:3], m2_hex[0:3]])
        plot_dof_vector(axes['xyz'], range(6), xyz_values, 
                       dataset_idx, dataset_width, color, marker)
        
        # RxRy: Cam rx,ry then M2 rx,ry (convert to arcsec)
        rxry_values = np.concatenate([cam_hex[3:5], m2_hex[3:5]]) * 3600
        plot_dof_vector(axes['rxry'], range(4), rxry_values,
                       dataset_idx, dataset_width, color, marker)
        
        # Bending modes
        plot_dof_vector(axes['m1m3_bends'], range(20), m1m3_bends,
                       dataset_idx, dataset_width, color, marker)
        plot_dof_vector(axes['m2_bends'], range(20), m2_bends,
                       dataset_idx, dataset_width, color, marker)
    
    finalize_dof_figure(fig, axes, file_keys, dataset_colors, 
                       title, output_path)


def finalize_dof_figure(fig, axes, file_keys, dataset_colors, 
                        title, output_path, marker_size=6):
    """Add legend, title, and save DOF figure.
    
    Parameters
    ----------
    fig : Figure
    axes : dict of Axes
    file_keys : list of str
    dataset_colors : sequence of colors
    title : str
    output_path : Path
    marker_size : float
    """
    # Add legend to one subplot
    handles, labels = [], []
    for file_idx, file_key in enumerate(file_keys):
        color = dataset_colors[file_idx]
        marker = 'o'
        handles.append(plt.Line2D([0], [0], color=color, marker=marker,
                                  linestyle='none', markersize=marker_size))
        labels.append(file_key)
    
    axes['m2_bends'].legend(handles, labels, ncols=1, loc='upper right',
                           fontsize=9)
    
    fig.suptitle(title, fontsize=12)
    
    print(f"  Saving DOF plot to {output_path}")
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

### Calculate DZs and DOFs

In [ ]:
dz_arr_list = median_per_group(rot_groups, n_focal, n_pupil)
dof_hat_list = [] # DOF vectors
rec_dz_list = [] # reconstructed DZs from smatrix * DOF vector
d_dz_list = [] # DZ residuals
for i, (rotang, dz_data) in enumerate(zip(rotang_labels, dz_arr_list)):

    dz_flat = dz_data.reshape(-1)
    dof_hat, residuals, rank, s = np.linalg.lstsq(A, dz_flat, rcond=1e-3)
    dof_hat_list.append(dof_hat)

    dz_reconstructed = A @ dof_hat
    rec_dz_list.append(dz_reconstructed)
    dz_residual = dz_flat - dz_reconstructed
    d_dz_list.append(dz_residual)

### Plot all the things

In [ ]:
outdir = 'dk_to_dof_results/'
plot_version = 'v1'
colors = plt.cm.tab10.colors[:len(rotang_labels)]

In [ ]:
plot_dof_datasets(
    dof_hat_list,
    rotang_labels,
    colors,
    'DOF from median DZ coeffs',
    f'{outdir}dof_plot_{plot_version}.pdf'
)

In [ ]:
colors = plt.cm.tab10.colors[:len(rotang_labels)]

plot_dz_datasets(dz_arr_list,
                 pupil_indices,
                 rotang_labels,
                 colors,
                'DZ Coefficients',
                f'{outdir}dz_plot_{plot_version}.pdf'
                )

In [ ]:
plot_dz_datasets([rec_dz_list[i].reshape(6,21) for i in range(len(rec_dz_list))],
                 pupil_indices,
                 [rotang_labels[i] for i in range(len(rec_dz_list))],
                 colors,
                'Reconstructed DZ Coefficients',
                f'{outdir}dz_reconstruct_plot_{plot_version}.pdf'
                )

In [ ]:
plot_dz_datasets([d_dz_list[i].reshape(6,21) for i in range(len(rec_dz_list))],
                 pupil_indices,
                 [rotang_labels[i] for i in range(len(rec_dz_list))],
                 colors,
                'DZ Coefficient Residuals',
                f'{outdir}dz_residual_plot_{plot_version}.pdf'
                )